# NIH Chest X-ray: Tabular Baseline Model

### Project Objective

Build a baseline machine learning model to predict one (1) selected chest X-ray finding using patient and image metadata from the NIH ChestX-ray14 dataset.

### Project Workflow

1. **Define prediction target** -- Select a chest X-ray finding as prediction target, based on its prevalence and suitability for a baseline model.
2. **Prepare target label** -- Inspect the dataset and target labels. Convert multi-label "Finding Labels" column into a binary target: 1 if selected finding is present and 0 otherwise.
3. **Build metadata feature set** -- Use Age, Gender, View Position, and Follow-up number as initial features.
4. **Address class imbalance and data leakage** -- Inspect the class distribution, use stratified validation with class weighting. Ensure images from the same patient do not appear in both training and validation sets.
5. **Train and evaluate a baseline model** -- Use a scikit-learn pipeline with a baseline classifier, and evaluate it using ROC-AUC, precision-recall, and other appropriate metrics beyond accuracy.
6. **Interpret the model** -- Use SHAP or other feature importance methods to explore how metadata features relate to predictions, with a short write-up on clinical interpretation and discuss limitations.

### Important Note

This model uses metadata rather than actual X-ray images. It explores associations between metadata and dataset labels -- it does not diagnose disease from medical images.

## 01. Load Data

In [1]:
# Import libraries
import pandas as pd
import numpy as np
from pathlib import Path

# Load data
df = pd.read_csv("/Users/hweeenheah/Downloads/EDA_Resources/nih-chestxray-ml/sql-eda/Data_Entry_2017.csv")
print("Dataset shape:", df.shape)
print(df.head())

print("Libraries Imported. Set up Complete.")

Dataset shape: (112120, 12)
        Image Index          Finding Labels  Follow-up #  Patient ID  \
0  00000001_000.png            Cardiomegaly            0           1   
1  00000001_001.png  Cardiomegaly|Emphysema            1           1   
2  00000001_002.png   Cardiomegaly|Effusion            2           1   
3  00000002_000.png              No Finding            0           2   
4  00000003_000.png                  Hernia            0           3   

   Patient Age Patient Gender View Position  OriginalImage[Width  Height]  \
0           58              M            PA                 2682     2749   
1           58              M            PA                 2894     2729   
2           58              M            PA                 2500     2048   
3           81              M            PA                 2500     2048   
4           81              F            PA                 2582     2991   

   OriginalImagePixelSpacing[x     y]  Unnamed: 11  
0                      

## 02. Define prediction target

In [4]:
# Count unique label combinations
label_counts= df['Finding Labels'].value_counts()
print(label_counts.head(20))

Finding Labels
No Finding                           60361
Infiltration                          9547
Atelectasis                           4215
Effusion                              3955
Nodule                                2705
Pneumothorax                          2194
Mass                                  2139
Effusion|Infiltration                 1603
Atelectasis|Infiltration              1350
Consolidation                         1310
Atelectasis|Effusion                  1165
Pleural_Thickening                    1126
Cardiomegaly                          1093
Emphysema                              892
Infiltration|Nodule                    829
Atelectasis|Effusion|Infiltration      737
Fibrosis                               727
Edema                                  628
Cardiomegaly|Effusion                  484
Consolidation|Infiltration             441
Name: count, dtype: int64
